# Paper 4: constrained JSON failure replay

This notebook replays **nine selected prior format failures from A-OKVQA TRAIN development**. It freezes the original images, questions, effective entity lists and captured Wikipedia/Wikidata facts, then changes only the opted-in JSON decoder and its recorded likelihood definition. **`benchmark_result=False`; model training, calibration fitting, threshold selection and held-out evaluation are not performed.** Selection by prior failure cannot support an accuracy estimate.

Use a CUDA GPU runtime. In the **Files pane**, upload `JB_Paper4_JSON_Failure_Replay_Inputs_20261009.zip` into `/content/`. No Drive mount, account login or paid compute is needed by this notebook. Public pinned model downloads are separate from knowledge retrieval. Read cells in order; **do not use Run All**: launch cells return immediately, and monitor cells must be repeated until their jobs finish. A clean full software suite precedes the one-case GPU gate. Only a validated gate permits the nine-case diagnostic.

Every child writes full stdout/stderr and its exit status to disk. A monitor interruption does not deliberately kill or relaunch the child. Download the final archive even when a gate fails. Colab storage is temporary; a terminated runtime cannot recover its lost files. Strict invalid outputs stay invalid. A structurally valid answer is not evidence of factual correctness.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, os, platform, shutil, stat, subprocess, sys, time, uuid, zipfile
import xml.etree.ElementTree as ET

SOURCE_COMMIT = '8377ce77b2f1b1f8851ff6e72f36ccbd2902b7bd'
REPOSITORY = 'https://github.com/junnubabu-ctrl/paper4-selective-kbvqa.git'
SOURCE_BRANCH = 'automation/structured-json-20261009'
INPUT_ZIP_NAME = 'JB_Paper4_JSON_Failure_Replay_Inputs_20261009.zip'
INPUT_ZIP_SHA256 = '6284f522239d728e262ed4703e96677fa97ea95a00577eff2b28bcc454305b7b'
INPUT_ZIP_BYTES = 1544334
LISTING_SHA256 = '58d7712e9d9f6a945d42c376f1730e374d611a993e1d8a6ef3df4af325b666f6'
MODEL_LOCK_SHA256 = '25ca99d3b87f9828001c76a9c7919a6ecfbeb725429de0fd291bcbc0824b17a8'
EXPECTED_MODEL_REVISIONS = {
    'generator': '66285546d2b821cf421d4f5eb2576359d3770cd3',
    'qwen': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
}
EXPECTED_QUESTION_IDS = [
    'eXZA9iZoEcQPqqEcugfbS3', 'JFMzV2UcGc7QtjJVwkvhzz', 'BFJt3qcPR2GyxXXY87S9FE',
    '36XKFSRexU37GWH5qVpCmn', 'LsfKQG9jYSVB9F5tZcm3Li', '8WpVyyo4f5xCqiVASSLNMU',
    '5NgdUhemLptNMHSvCfZPbj', 'QyCxfwBGsigZKWsB5QqLUJ', 'X7zGQJcB4higpU9gnhpVXt',
]
# Change this tag only when deliberately starting a wholly new run.
RUN_TAG = 'json_replay_8377_20261009'
assert RUN_TAG.replace('_', '').isalnum()
RUN_ROOT = Path('/content/JB_Paper4') / RUN_TAG
REPO = Path('/content') / ('paper4_code_' + RUN_TAG)
INPUTS = RUN_ROOT / 'inputs'
GATE_DIR = RUN_ROOT / 'gate1'
DIAGNOSTIC_DIR = RUN_ROOT / 'diagnostic9'
JOBS = RUN_ROOT / 'jobs'
if '_SESSION_ROOT' not in globals():
    assert not RUN_ROOT.exists() and not REPO.exists(), 'Existing run/code folder: use a new declared RUN_TAG; do not mix or overwrite prior records.'
    RUN_ROOT.mkdir(parents=True)
    JOBS.mkdir()
    _SESSION_ROOT = str(RUN_ROOT)
else:
    assert _SESSION_ROOT == str(RUN_ROOT), 'Restart initialization with a fresh runtime/tag before changing paths.'

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.part')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.write('\n')
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)

def audit_event(name, value):
    event = {'event': name, 'utc': utc_now(), 'source_commit': SOURCE_COMMIT, **value}
    with (RUN_ROOT / 'audit_events.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False, allow_nan=False) + '\n')
        stream.flush()
        os.fsync(stream.fileno())
    save_json(RUN_ROOT / (name + '_latest.json'), event)
    return event

def pid_identity(pid):
    try:
        fields = Path(f'/proc/{int(pid)}/stat').read_text().rsplit(')', 1)[1].split()
        return fields[0], fields[19]
    except (OSError, ValueError, IndexError):
        return None

def job_record(label):
    return json.loads((JOBS / label / 'job.json').read_text())

def job_is_live(record):
    if record.get('status') not in {'LAUNCHING', 'RUNNING'}:
        return False
    identity = record
    if 'worker_pid' not in identity:
        fallback = JOBS / record['label'] / 'launcher_process.json'
        if not fallback.exists():
            return True  # conservatively block another launch while identity is pending
        identity = json.loads(fallback.read_text())
    found = pid_identity(identity['worker_pid'])
    return found is not None and found[0] != 'Z' and found[1] == identity.get('worker_start_ticks')

def active_jobs():
    return [json.loads(p.read_text()) for p in JOBS.glob('*/job.json')
            if job_is_live(json.loads(p.read_text()))]

print(json.dumps({'run_root': str(RUN_ROOT), 'repository': REPOSITORY, 'pinned_commit': SOURCE_COMMIT,
    'input_zip': '/content/' + INPUT_ZIP_NAME, 'benchmark_result': False, 'model_training': False}, indent=2))


## 1. Retained asynchronous jobs

The worker runs the exact listed command without a shell and saves its terminal status independently of notebook monitoring. Each job label can launch once, and only one child may be active. If a launch cell is rerun, it displays the existing job instead of duplicating it. Run the next monitor cell repeatedly while `status` is `RUNNING`. `UNKNOWN_WORKER_EXIT` is a failure to retain a confirmed exit status; archive it and use a fresh run for another attempt.


In [ ]:
WORKER = RUN_ROOT / 'retained_job_worker.py'
worker_source = r"""
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys, time

def now():
    return datetime.now(timezone.utc).isoformat()
def save(path, row):
    temporary = path.with_name(path.name + '.part')
    with temporary.open('w', encoding='utf-8') as f:
        json.dump(row, f, indent=2, ensure_ascii=False, allow_nan=False)
        f.write('\n'); f.flush(); os.fsync(f.fileno())
    os.replace(temporary, path)
path = Path(sys.argv[1])
row = json.loads(path.read_text())
fields = Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()
row.update(status='RUNNING', worker_pid=os.getpid(), worker_start_ticks=fields[19], started_utc=now())
save(path, row)
started = time.monotonic()
try:
    with Path(row['stdout_path']).open('wb') as stdout, Path(row['stderr_path']).open('wb') as stderr:
        row['steps'] = []
        for command in row['commands']:
            result = subprocess.run(command, cwd=row['cwd'], stdout=stdout, stderr=stderr,
                env={**os.environ, 'PYTHONUNBUFFERED': '1', 'HF_HUB_DISABLE_IMPLICIT_TOKEN': '1',
                     'HF_HUB_DISABLE_TELEMETRY': '1'})
            row['steps'].append({'argv': command, 'returncode': result.returncode})
            save(path, row)
            if result.returncode != 0:
                break
    row.update(status='FINISHED', returncode=result.returncode)
except BaseException as error:
    row.update(status='WORKER_FAILED', returncode=None, error=f'{type(error).__name__}: {error}')
finally:
    row.update(finished_utc=now(), elapsed_s=time.monotonic()-started)
    save(path, row)
"""
if WORKER.exists():
    assert WORKER.read_text() == worker_source
else:
    WORKER.write_text(worker_source, encoding='utf-8')

def launch_job(label, commands, *, cwd='/content', out_dir=None):
    assert label.replace('_', '').isalnum()
    folder = JOBS / label
    if folder.exists():
        print(json.dumps(job_record(label), indent=2))
        print('Existing job retained; no duplicate command launched.')
        return
    assert not active_jobs(), 'A child is active; monitor it before launching another job.'
    if out_dir is not None:
        assert not Path(out_dir).exists(), 'GPU output directory must be fresh.'
    folder.mkdir()
    record = {'schema': 'paper4-retained-async-job-v1', 'label': label, 'status': 'LAUNCHING',
        'commands': [[str(value) for value in command] for command in commands],
        'cwd': str(cwd), 'source_commit': SOURCE_COMMIT, 'created_utc': utc_now(),
        'stdout_path': str(folder / 'stdout.log'), 'stderr_path': str(folder / 'stderr.log'),
        'out_dir': str(out_dir) if out_dir is not None else None, 'returncode': None}
    path = folder / 'job.json'
    save_json(path, record)
    try:
        with (folder / 'worker.log').open('wb') as supervisor:
            process = subprocess.Popen([sys.executable, str(WORKER), str(path)], cwd='/content',
                stdout=supervisor, stderr=subprocess.STDOUT, start_new_session=True)
    except BaseException as error:
        record.update(status='LAUNCH_FAILED', error=f'{type(error).__name__}: {error}')
        save_json(path, record)
        raise
    found = pid_identity(process.pid)
    save_json(folder / 'launcher_process.json', {'worker_pid': process.pid,
        'worker_start_ticks': found[1] if found is not None else None})
    # Wait only for the worker to establish its independently retained identity.
    deadline = time.monotonic() + 3
    while time.monotonic() < deadline and job_record(label)['status'] == 'LAUNCHING':
        time.sleep(0.05)
    print(json.dumps(job_record(label), indent=2))

def monitor_job(label, seconds=30):
    assert 0 <= seconds <= 30
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline and job_is_live(job_record(label)):
        time.sleep(1)
    record = job_record(label)
    if record['status'] in {'RUNNING', 'LAUNCHING'} and not job_is_live(record):
        record.update(status='UNKNOWN_WORKER_EXIT', observed_utc=utc_now())
        save_json(JOBS / label / 'job.json', record)
    for kind in ['stdout', 'stderr']:
        path = Path(record[kind + '_path'])
        if path.exists() and path.stat().st_size:
            print('[' + kind + ' retained tail]\n' + path.read_bytes()[-6000:].decode('utf-8', errors='replace'))
    output = record.get('out_dir')
    counts = {name: len(list((Path(output) / name).glob('*.json'))) for name in
              ['contexts', 'candidates', 'critics']} if output else {}
    print(json.dumps({'label': label, 'status': record['status'], 'returncode': record['returncode'],
        'record_counts': counts, 'full_logs': str(JOBS / label)}, indent=2))
    return record

def require_finished(label, allowed=(0,)):
    row = job_record(label)
    assert row['status'] == 'FINISHED', f'{label}: repeat its monitor cell until FINISHED; current={row["status"]}'
    assert row['returncode'] in allowed, f'{label} failed with exit {row["returncode"]}; retain logs and use the archive cell.'
    return row


## 2. Verify the uploaded frozen inputs

Upload through the Files pane first. The ZIP must be exactly **1,544,334 bytes** with the pinned SHA256. Extraction rejects duplicate names, traversal and symlinks. It checks the independently pinned checksum listing and every one of its **53 member hashes**, before and after extraction. Historical generated answers in `audit_source/` are audit-only; inference receives only the manifest, original images and frozen question-indexed facts. The attestation inherits archived TRAIN provenance, without asserting a signed upstream authenticity or licensing proof.


In [ ]:
UPLOADED_ZIP = Path('/content') / INPUT_ZIP_NAME
assert UPLOADED_ZIP.is_file(), 'Use the Files pane to upload the exact named input ZIP into /content/.'
observed = {'path': str(UPLOADED_ZIP), 'bytes': UPLOADED_ZIP.stat().st_size, 'sha256': sha256_file(UPLOADED_ZIP)}
retained_zip = RUN_ROOT / INPUT_ZIP_NAME
if not retained_zip.exists():
    shutil.copy2(UPLOADED_ZIP, retained_zip)
assert sha256_file(retained_zip) == observed['sha256'], 'Retained and uploaded inputs differ.'
audit_event('input_upload', observed)
assert observed['bytes'] == INPUT_ZIP_BYTES and observed['sha256'] == INPUT_ZIP_SHA256, 'Input ZIP identity mismatch: no inference permitted.'
with zipfile.ZipFile(retained_zip) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) == 54 and archive.testzip() is None
    for item in archive.infolist():
        path = PurePosixPath(item.filename)
        assert not path.is_absolute() and '..' not in path.parts and '\\' not in item.filename
        assert not item.is_dir() and not stat.S_ISLNK(item.external_attr >> 16)
    assert hashlib.sha256(archive.read('file_sha256.json')).hexdigest() == LISTING_SHA256
    listing = json.loads(archive.read('file_sha256.json'))
    assert listing['algorithm'] == 'SHA256' and len(listing['files']) == 53
    assert set(names) == set(listing['files']) | {'file_sha256.json'}
    for name, expected in listing['files'].items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected, name
    if not INPUTS.exists():
        INPUTS.mkdir()
        archive.extractall(INPUTS)
    assert {str(p.relative_to(INPUTS)) for p in INPUTS.rglob('*') if p.is_file()} == set(names)
    assert sha256_file(INPUTS / 'file_sha256.json') == LISTING_SHA256
    for name, expected in listing['files'].items():
        assert sha256_file(INPUTS / name) == expected, name
INPUTS_VERIFIED = True
audit_event('input_integrity', {'passed': True, 'zip_bytes': INPUT_ZIP_BYTES,
    'zip_sha256': INPUT_ZIP_SHA256, 'listing_sha256': LISTING_SHA256,
    'verified_member_hashes': 53, 'knowledge_downloads': False, 'benchmark_result': False})
print('PASS: exact ZIP, independent listing, and all 53 member hashes verified.')


## 3. Clone the pinned source and install dependencies

The public branch supplies the exact detached commit below. Existing code folders are refused. Installation pins **Transformers 4.57.1** and **LM Format Enforcer 0.11.3**; model revisions remain the repository's original lock. Both installation and software checks run in child processes, avoiding stale notebook imports after package changes. No login is attempted.


In [ ]:
assert INPUTS_VERIFIED
launch_job('clone', [
    ['git', 'clone', '--no-checkout', '--branch', SOURCE_BRANCH, REPOSITORY, REPO],
    ['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT],
])


In [ ]:
monitor_job('clone')


In [ ]:
require_finished('clone')
def assert_pinned_checkout():
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == SOURCE_COMMIT, 'Source commit mismatch.'
    assert subprocess.run(['git', '-C', str(REPO), 'symbolic-ref', '-q', 'HEAD'], capture_output=True).returncode != 0
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'Tracked source changed.'
    return actual
assert_pinned_checkout()
assert sha256_file(REPO / 'configs/models_20260915.json') == MODEL_LOCK_SHA256
model_lock = json.loads((REPO / 'configs/models_20260915.json').read_text())
assert {name: model_lock[name]['revision'] for name in EXPECTED_MODEL_REVISIONS} == EXPECTED_MODEL_REVISIONS
snapshot = RUN_ROOT / 'source_snapshot'
snapshot.mkdir(exist_ok=True)
tracked = subprocess.check_output(['git', '-C', str(REPO), 'ls-files', '-z']).decode().split('\0')
source_hashes = {}
for name in filter(None, tracked):
    source = REPO / name
    if source.is_file():
        target = snapshot / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(source, target)
        assert sha256_file(target) == sha256_file(source)
        source_hashes[name] = sha256_file(source)
audit_event('source_identity', {'actual_commit': SOURCE_COMMIT, 'detached': True,
    'tracked_source_clean': True, 'source_files': source_hashes,
    'model_lock_sha256': MODEL_LOCK_SHA256,
    'models': {name: model_lock[name] for name in EXPECTED_MODEL_REVISIONS}})
launch_job('install', [[sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[vlm,dev]',
                       'transformers==4.57.1', 'lm-format-enforcer==0.11.3']], cwd=REPO)


In [ ]:
monitor_job('install')


## 4. Complete software checks before model inference

This runs the entire pinned suite, including the genuine HF prefix-callback test built from a local tokenizer with no downloads. Require all **215 tests**, no failures/errors and no skips. Software success is separate from actual Qwen GPU execution. The runtime record retains package versions, hardware, source and model identity; it does not claim minimum VRAM or answer correctness.


In [ ]:
require_finished('install')
assert_pinned_checkout()
packages = {name: importlib.metadata.version(name) for name in
    ['torch', 'transformers', 'lm-format-enforcer', 'tokenizers', 'accelerate', 'bitsandbytes',
     'numpy', 'scikit-learn', 'pillow', 'pytest']}
assert packages['transformers'] == '4.57.1' and packages['lm-format-enforcer'] == '0.11.3'
audit_event('installed_packages', {'packages': packages, 'python': sys.version, 'platform': platform.platform()})
launch_job('software_tests', [[sys.executable, '-m', 'pytest', 'tests', '-ra',
    '--junitxml=' + str(RUN_ROOT / 'software_tests.xml')]], cwd=REPO)


In [ ]:
monitor_job('software_tests')


In [ ]:
require_finished('software_tests')
suites = list(ET.parse(RUN_ROOT / 'software_tests.xml').getroot().iter('testsuite'))
test_counts = {key: sum(int(suite.attrib.get(key, 0)) for suite in suites)
               for key in ['tests', 'failures', 'errors', 'skipped']}
assert test_counts == {'tests': 215, 'failures': 0, 'errors': 0, 'skipped': 0}, test_counts
assert_pinned_checkout()
SOFTWARE_PASS = True
audit_event('software_gate', {'passed': True, 'counts': test_counts, 'benchmark_result': False,
    'model_inference_performed_by_this_cell': False})
import torch
assert torch.cuda.is_available(), 'CUDA GPU required; use the final archive cell to retain the software-only result.'
hardware = {'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'devices': [{'index': i, 'name': torch.cuda.get_device_name(i),
                 'total_memory_bytes': int(torch.cuda.get_device_properties(i).total_memory)}
                for i in range(torch.cuda.device_count())], 'benchmark_result': False}
audit_event('hardware', hardware)
print(json.dumps({'software_gate': 'PASS', **hardware}, indent=2))


## 5. Independent inference-boundary preflight

The actual CLI loader, frozen provider and CPU evidence filter validate all nine rows before GPU inference. All inference fields are scanned for forbidden reference/label keys. Original context objects, retrieved facts and selected evidence must match exactly. The replay preserves **effective** entity lists; three old inputs had empty lists that later became `fruit`, `pine tree` and `eat`. Historical raw generated answers are never passed as inference or repair targets. Zero selected evidence in one case is preserved.


In [ ]:
assert SOFTWARE_PASS and INPUTS_VERIFIED
assert_pinned_checkout()
import importlib.util
from dataclasses import asdict
from types import SimpleNamespace
sys.path.insert(0, str(REPO / 'src'))
from paper4_kbvqa.execution.fixed_candidate import (
    assert_no_gold_keys, digest, file_digest, read_verified, FrozenQuestionEvidence, inference_identity, validate_image, plain,
)
from paper4_kbvqa.filtering.relevance import RelevanceFilter
from paper4_kbvqa.types import Evidence
spec = importlib.util.spec_from_file_location('verified_pilot_cli', REPO / 'scripts/run_fixed_candidate_pilot.py')
pilot_cli = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pilot_cli)
EXPECTED_EXECUTION_SOURCE_IDENTITY = pilot_cli.source_code_identity()
samples, data_identity = pilot_cli.load_pilot_inputs(SimpleNamespace(pilot_dir=None,
    manifest=str(INPUTS / 'inference_manifest.jsonl'),
    train_development_attestation=str(INPUTS / 'train_development_attestation.json'), max_samples=9))
assert len(samples) == len({sample.question_id for sample in samples}) == 9
assert [sample.question_id for sample in samples] == EXPECTED_QUESTION_IDS, 'Selected case order changed.'
attestation = json.loads((INPUTS / 'train_development_attestation.json').read_text())
assert_no_gold_keys(attestation, 'train_development_attestation')
assert attestation['source_split'] == 'train' and attestation['partition_role'] == 'development_only'
assert attestation['benchmark_result'] is False and attestation['held_out_evaluation'] is False
assert attestation['question_count'] == 9 and attestation['distinct_image_count'] == 9
rows = [json.loads(line) for line in (INPUTS / 'frozen_question_evidence.jsonl').read_text().splitlines() if line.strip()]
assert len(rows) == 9 and [row['question_id'] for row in rows] == [sample.question_id for sample in samples]
frozen_rows = {}
for row in rows:
    assert_no_gold_keys(row, 'static_evidence')
    assert set(row) == {'question_id', 'evidence'}
    frozen_rows[row['question_id']] = [Evidence(**{**item, 'entity_ids': tuple(item.get('entity_ids', []))}) for item in row['evidence']]
provider = FrozenQuestionEvidence(frozen_rows)
ledger = json.loads((INPUTS / 'source_ledger.json').read_text())
assert ledger['source_set'] == ['wikipedia', 'wikidata'] and ledger['new_live_retrieval_performed'] is False
assert ledger['evidence_snapshot_sha256'] == sha256_file(INPUTS / 'frozen_question_evidence.jsonl')
preflight_records = []
for sample in samples:
    assert sample.metadata['source_split'] == 'train'
    image = validate_image(sample)
    identity = inference_identity(sample)
    assert_no_gold_keys(identity, 'inference_identity')
    stem = hashlib.sha256(sample.question_id.encode()).hexdigest()
    old_context = read_verified(INPUTS / 'audit_source/two_source_pilot50_v3/contexts' / (stem + '.json'))
    old_candidate = read_verified(INPUTS / 'audit_source/two_source_pilot50_v3/candidates' / (stem + '.json'))
    assert old_candidate['generator_result']['parser_status'] == 'invalid_json'
    retrieved = provider.retrieve_for_question_id(sample.question_id, limit=10)
    selected, _ = RelevanceFilter().select(retrieved, sample.question, list(sample.visual_entities), top_k=5)
    context = plain({'question': sample.question, 'visual_entities': list(sample.visual_entities),
        'ordered_evidence': [asdict(item) for item in selected], 'image_sha256': image['image_sha256']})
    assert context == old_context['context'] and digest(context) == old_context['context_sha256']
    assert plain([asdict(item) for item in retrieved]) == old_context['retrieved_evidence']
    preflight_records.append({'question_id': sample.question_id, 'image_sha256': image['image_sha256'],
        'selected_evidence_count': len(selected), 'context_sha256': digest(context), 'same_captured_context': True})
assert [row['selected_evidence_count'] for row in preflight_records] == [4, 3, 3, 4, 2, 3, 3, 0, 3]
PREFLIGHT_PASS = True
audit_event('inference_preflight', {'passed': True, 'data_identity': data_identity,
    'question_id_order': EXPECTED_QUESTION_IDS, 'records': preflight_records, 'gold_fields_used': False, 'knowledge_provider_http': False,
    'training': False, 'benchmark_result': False})
BASE_ARGV = [sys.executable, REPO / 'scripts/run_fixed_candidate_pilot.py',
    '--manifest', INPUTS / 'inference_manifest.jsonl',
    '--train-development-attestation', INPUTS / 'train_development_attestation.json',
    '--evidence-jsonl', INPUTS / 'frozen_question_evidence.jsonl',
    '--source-ledger', INPUTS / 'source_ledger.json', '--model-lock', REPO / 'configs/models_20260915.json',
    '--sources', 'wikipedia,wikidata', '--stage', 'train-development',
    '--retrieval-top-k', '10', '--filtered-top-k', '5', '--max-pixels', '1003520', '--seed', '2026',
    '--no-auto-entities', '--constrained-json']
print(json.dumps({'preflight': 'PASS', 'selected_cases': 9, 'selected_evidence_counts': [row['selected_evidence_count'] for row in preflight_records],
    'common_argv': [str(value) for value in BASE_ARGV]}, indent=2))


## 6. One-case GPU gate and exact output audit

Launch one selected case in its own fresh folder. Model downloads and loading can take several minutes. The gate requires one fully archived valid context/candidate/critic triplet, a checksum-verified final summary, exit 0, the frozen source identity, exact old/new context, and the recorded unprocessed-logit likelihood source. Unknown and repeated citation IDs remain diagnostics. No evidence is substituted and no raw output is repaired.


In [ ]:
assert SOFTWARE_PASS and PREFLIGHT_PASS and INPUTS_VERIFIED
assert_pinned_checkout()
launch_job('gpu_gate1', [BASE_ARGV + ['--max-samples', '1', '--out-dir', GATE_DIR]], cwd=REPO, out_dir=GATE_DIR)


In [ ]:
monitor_job('gpu_gate1')


In [ ]:
def audit_run(output, expected_count, job_label):
    report = {'output_dir': str(output), 'expected_count': expected_count, 'status': 'NOT_STARTED',
        'audit_pass': False, 'errors': [], 'records': [], 'benchmark_result': False,
        'training': False, 'calibration_fit': False, 'threshold_selection': False}
    if not output.exists():
        return report
    try:
        assert not active_jobs(), 'A child is still active.'
        job = job_record(job_label)
        assert job['status'] == 'FINISHED', 'Confirmed terminal process status unavailable.'
        report['returncode'] = job['returncode']
        assert job['returncode'] in [0, 2], 'Infrastructure failure: inspect complete logs.'
        assert not (output / 'source_cache').exists() and not (output / 'source_capture.jsonl').exists()
        # Verify every checksummed JSON envelope, including invalid records.
        for path in output.rglob('*.json'):
            read_verified(path)
        run = read_verified(output / 'run_identity.json')
        run_sha = digest(run)
        assert run['stage'] == 'train-development' and len(run['inputs']) == expected_count
        plan = run['run_config']['plan']
        assert plan['constrained_json'] is True and plan['source_mode'] == 'static_evidence_alternative'
        assert plan['sources'] == ['wikipedia', 'wikidata'] and plan['benchmark_result'] is False
        assert plan['execution_config']['auto_extract_entities'] is False
        assert run['source_identity']['mode'] == 'static_evidence_alternative'
        assert run['source_identity']['evidence_snapshot_sha256'] == sha256_file(INPUTS / 'frozen_question_evidence.jsonl')
        assert run['source_identity']['source_ledger_sha256'] == sha256_file(INPUTS / 'source_ledger.json')
        assert run['run_config']['model_lock_sha256'] == MODEL_LOCK_SHA256
        assert run['run_config']['code_identity'] == EXPECTED_EXECUTION_SOURCE_IDENTITY
        assert run['run_config']['runtime']['packages']['transformers'] == '4.57.1'
        assert run['run_config']['runtime']['packages']['lm-format-enforcer'] == '0.11.3'
        selected_samples = samples[:expected_count]
        assert run['inputs'] == [inference_identity(sample) for sample in selected_samples]
        stems = {hashlib.sha256(sample.question_id.encode()).hexdigest() for sample in selected_samples}
        paths = {name: {p.stem: p for p in (output / name).glob('*.json')} for name in ['contexts', 'candidates', 'critics']}
        assert all(set(group) == stems for group in paths.values()), 'Missing or foreign question triplets.'
        matched = []
        for sample in selected_samples:
            stem = hashlib.sha256(sample.question_id.encode()).hexdigest()
            context, candidate, critic = [read_verified(paths[name][stem]) for name in ['contexts', 'candidates', 'critics']]
            old_context = read_verified(INPUTS / 'audit_source/two_source_pilot50_v3/contexts' / (stem + '.json'))
            old_candidate = read_verified(INPUTS / 'audit_source/two_source_pilot50_v3/candidates' / (stem + '.json'))
            for record in [context, candidate, critic]:
                assert record['question_id'] == sample.question_id and record['run_identity_sha256'] == run_sha
            assert context['stage'] == candidate['stage'] == 'train-development'
            assert context['input_identity'] == candidate['input_identity'] == inference_identity(sample)
            assert context['source_identity'] == run['source_identity']
            assert context['context'] == old_context['context']
            assert context['context_sha256'] == old_context['context_sha256'] == digest(context['context'])
            assert context['retrieved_evidence'] == context['retrieved_pool'] == old_context['retrieved_evidence']
            assert context['retrieved_evidence'] == plain([asdict(item) for item in frozen_rows[sample.question_id][:10]])
            assert 'visual_grounding_record' not in context
            assert candidate['context_record_sha256'] == digest(context)
            assert critic['candidate_record_sha256'] == digest(candidate)
            generated = candidate.get('generator_result', {})
            old_generated = old_candidate['generator_result']
            if generated:
                identity = generated['generation_identity']
                old_identity = old_generated['generation_identity']
                for field in ['question', 'ordered_evidence', 'rendered_prompt', 'prompt_version', 'image_sha256',
                    'image_dimensions_passed_to_processor', 'image_grid_thw', 'processed_image_dimensions',
                    'prompt_token_count', 'seed', 'four_bit', 'max_pixels', 'model_id', 'requested_revision', 'resolved_revision']:
                    assert identity[field] == old_identity[field], 'Changed generator input/config field: ' + field
                assert identity['decoding']['max_new_tokens'] == 192 and identity['decoding']['do_sample'] is False
                assert identity['decoding']['output_logits'] is True and identity['decoding']['output_scores'] is False
                assert generated['likelihood_score_source'] == 'transformers_generate_unprocessed_logits_v1'
                assert identity['decoding']['likelihood_score_source'] == generated['likelihood_score_source']
                constraint = identity['decoding']['json_constraint']
                assert constraint['library']['version'] == '0.11.3'
                assert constraint['schema_name'] == 'answer_with_evidence_ids'
                assert 'enum' not in json.dumps(constraint['schema'])
                assert constraint['tokenizer']['clean_up_tokenization_spaces'] is False
                assert generated['raw_text'] == generated['raw_output']
                assert len(generated['generated_token_ids']) == len(generated['generated_token_logprobs'])
                if generated['raw_confidence_status'] == 'finite':
                    import math
                    logps = generated['generated_token_logprobs']
                    assert logps and all(math.isfinite(value) for value in logps)
                    assert math.isclose(generated['raw_confidence'], math.exp(sum(logps) / len(logps)), rel_tol=1e-12)
                if generated['parsed_ok']:
                    assert generated['parser_status'] == 'valid'
                else:
                    assert candidate['candidate_eligible'] is False
            if candidate.get('candidate_eligible'):
                assert candidate['context'] == context['context']
                assert candidate['context_sha256'] == context['context_sha256']
                assert candidate['candidate_sha256'] == digest(candidate['candidate'])
                assert candidate['candidate']['generator_result_sha256'] == digest(generated)
            if critic['status'] == 'VALID_MATCHED_DEVELOPMENT_RECORD':
                for variant in ['B2', 'B3']:
                    result = critic[variant]
                    assert result['answer'] == candidate['candidate']['answer']
                    assert result['candidate_sha256'] == candidate['candidate_sha256']
                    assert result['context_sha256'] == candidate['context_sha256']
                    assert result['calibrated_confidence'] is result['threshold'] is result['abstain'] is None
                assert critic['benchmark_result'] is False and critic['calibration_implemented'] is False
            selected_ids = {item['evidence_id'] for item in context['context']['ordered_evidence']}
            raw_ids = generated.get('raw_evidence_ids', [])
            assert isinstance(raw_ids, list) and all(isinstance(value, str) for value in raw_ids)
            unknown = [value for value in raw_ids if value not in selected_ids]
            repeated = [value for value in dict.fromkeys(raw_ids) if raw_ids.count(value) > 1]
            report['records'].append({'question_id': sample.question_id, 'question': sample.question,
                'candidate_status': candidate['status'], 'critic_status': critic['status'],
                'candidate_eligible': candidate.get('candidate_eligible'), 'answer': generated.get('answer'),
                'raw_text': generated.get('raw_text'), 'generated_token_ids': generated.get('generated_token_ids'),
                'generated_token_logprobs': generated.get('generated_token_logprobs'),
                'raw_confidence': generated.get('raw_confidence'), 'parser_status': generated.get('parser_status'),
                'generation_termination': generated.get('generation_termination'),
                'selected_evidence_count': len(selected_ids), 'same_captured_context': True,
                'unknown_citation_occurrences': unknown, 'repeated_citation_values': repeated,
                'critic_result': critic.get('critic_result'),
                'errors': {name: value['errors'] for name, value in [('context', context), ('candidate', candidate), ('critic', critic)]}})
            matched.append({'question_id': sample.question_id, 'context_record_sha256': digest(context),
                'candidate_record_sha256': digest(candidate), 'critic_record_sha256': digest(critic),
                'candidate_status': candidate['status'], 'critic_status': critic['status'],
                'B2': critic.get('B2'), 'B3': critic.get('B3')})
        summary = read_verified(output / 'pilot_summary.json')
        assert summary['run_identity_sha256'] == run_sha and summary['matched'] == matched
        valid = sum(row['critic_status'] == 'VALID_MATCHED_DEVELOPMENT_RECORD' for row in report['records'])
        counts = {'requested': expected_count, 'attempted': len(report['records']), 'valid': valid,
                  'invalid': len(report['records']) - valid, 'unattempted': expected_count - len(report['records'])}
        assert all(summary[field] == value for field, value in counts.items())
        assert summary['benchmark_result'] is False and summary['scientific_claims_validated'] is False
        assert job['returncode'] == (0 if counts['invalid'] == 0 else 2)
        report.update(status=summary['status'], audit_pass=True, run_identity_sha256=run_sha,
            data_identity=run['run_config']['data_identity'], **counts)
    except Exception as error:
        report.update(status='AUDIT_FAILED', audit_pass=False)
        report['errors'].append(f'{type(error).__name__}: {error}')
    return report

require_finished('gpu_gate1', allowed=(0, 2))
gate_audit = audit_run(GATE_DIR, 1, 'gpu_gate1')
ONE_GATE_PASS = (gate_audit['audit_pass'] and gate_audit.get('returncode') == 0
    and gate_audit.get('requested') == gate_audit.get('attempted') == gate_audit.get('valid') == 1
    and gate_audit.get('invalid') == gate_audit.get('unattempted') == 0)
audit_event('one_case_gate', {'passed': ONE_GATE_PASS, 'audit': gate_audit})
print(json.dumps(gate_audit, indent=2, ensure_ascii=False))
print('GPU gate:', 'PASS' if ONE_GATE_PASS else 'BLOCKED; skip diagnostic launch and retain the failure archive.')


## 7. Nine-case diagnostic

Only a currently revalidated one-case gate permits this launch. The nine-case run includes that same case again; it is a separate replay, not ten new unique benchmark cases. **`--continue-on-invalid`** is the pinned CLI flag. It attempts the bounded inputs while retaining every invalid record; exit **2** remains an invalid diagnostic outcome. Missing summaries or incomplete triplets remain infrastructure failures. Never combine these selected counts with benchmark accuracy.


In [ ]:
if ONE_GATE_PASS:
    assert SOFTWARE_PASS and PREFLIGHT_PASS and INPUTS_VERIFIED
    assert_pinned_checkout()
    current_gate = audit_run(GATE_DIR, 1, 'gpu_gate1')
    assert current_gate['audit_pass'] and current_gate['valid'] == current_gate['requested'] == 1
    assert current_gate['invalid'] == current_gate['unattempted'] == 0 and current_gate['returncode'] == 0
    launch_job('gpu_diagnostic9', [BASE_ARGV + ['--max-samples', '9', '--continue-on-invalid',
        '--out-dir', DIAGNOSTIC_DIR]], cwd=REPO, out_dir=DIAGNOSTIC_DIR)
else:
    print('SKIPPED: one-case gate blocked; all failure evidence remains retained.')


In [ ]:
if (JOBS / 'gpu_diagnostic9/job.json').exists():
    monitor_job('gpu_diagnostic9')
else:
    print('Nine-case diagnostic was not launched.')


## 8. Actual execution audit and printed outputs

The audit reads immutable checksummed records and reconciles actual requested, attempted, valid, invalid and unattempted counts. It prints the questions, raw generated answers, strict parser/termination statuses, citations, critic diagnostics and errors. All complete output records remain in the archive. Whole-response token likelihood includes JSON, citation and EOS tokens; it is **uncalibrated**, comes from the original unprocessed model logits, and is not numerically equivalent to legacy processed scores. No reference-answer accuracy, calibration or selective policy is computed.


In [ ]:
assert not active_jobs(), 'A child remains active: repeat its monitor cell before final audit.'
runs = []
if 'audit_run' in globals():
    runs = [audit_run(GATE_DIR, 1, 'gpu_gate1'), audit_run(DIAGNOSTIC_DIR, 9, 'gpu_diagnostic9')]
execution_audit = {'runs': runs, 'benchmark_result': False, 'training': False,
    'held_out_evaluation': False, 'calibration_fit': False, 'threshold_selection': False,
    'selection': 'Nine archived TRAIN-development strict answer invalid_json failures in original order',
    'one_case_gate_passed': globals().get('ONE_GATE_PASS', False),
    'software_gate_passed': globals().get('SOFTWARE_PASS', False),
    'input_preflight_passed': globals().get('PREFLIGHT_PASS', False)}
if len(runs) == 2 and all(run.get('audit_pass') for run in runs):
    execution_audit['same_cross_run_data_identity'] = runs[0]['data_identity'] == runs[1]['data_identity']
    assert execution_audit['same_cross_run_data_identity']
audit_event('final_execution_audit', execution_audit)
for run in runs:
    print(json.dumps({key: value for key, value in run.items() if key != 'records'}, indent=2))
    for row in run['records']:
        print(json.dumps(row, ensure_ascii=False))
print('Execution validity and format recovery counts are not factual correctness or benchmark accuracy.')


## 9. Download everything, including failures

Use this cell after all active children end, including when any earlier validation failed. It retains the uploaded input ZIP, every extracted input and historical audit record, exact tracked source snapshot, package/hardware records, full command logs/statuses, raw valid and invalid model outputs, critic/context records, resource sessions and append-only audit events. It inventories and hashes every retained file before making the ZIP. Partial files are retained and labeled in the inventory; they do not establish completed execution. The archive SHA256 detects changes, without asserting upstream authenticity or licensing.

Further research requires broader development checks, evidence relevance/provenance review, appropriate source/license freezing, image-disjoint calibration and threshold subsets, locked policies, held-out matched comparisons and failure analysis. JSON recovery alone does not establish manuscript readiness or PhD completion.


In [ ]:
assert not active_jobs(), 'Archive only after the existing child ends; do not launch another process.'
inventory = [{'path': str(path.relative_to(RUN_ROOT)), 'bytes': path.stat().st_size,
    'sha256': sha256_file(path), 'partial_file': path.name.endswith(('.part', '.partial'))}
    for path in sorted(RUN_ROOT.rglob('*')) if path.is_file()
    and path.name != 'retained_artifact_inventory.json']
save_json(RUN_ROOT / 'retained_artifact_inventory.json', inventory)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
archive_path = Path('/content') / ('JB_Paper4_JSON_Failure_Replay_Results_' + stamp + '.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, str(path.relative_to(RUN_ROOT)))
archive_info = {'path': str(archive_path), 'bytes': archive_path.stat().st_size,
    'sha256': sha256_file(archive_path), 'source_commit': SOURCE_COMMIT,
    'benchmark_result': False, 'training': False}
print(json.dumps(archive_info, indent=2))
Path(str(archive_path) + '.sha256').write_text(archive_info['sha256'] + '  ' + archive_path.name + '\n')
from google.colab import files
files.download(str(archive_path))
files.download(str(archive_path) + '.sha256')


Implementation references: [Transformers 4.57.1 generation outputs](https://huggingface.co/docs/transformers/v4.57.1/en/internal/generation_utils) distinguish unprocessed `logits` from processed `scores`; [LM Format Enforcer 0.11.3](https://pypi.org/project/lm-format-enforcer/0.11.3/) supplies the direct prefix callback. This notebook is pinned to executable source commit `8377ce77b2f1b1f8851ff6e72f36ccbd2902b7bd`. Notebook validation checks Python syntax and declared guards; this blank reusable notebook itself contains no executed GPU results.
